# 💼 薪資預測 Gradio Web UI 實作教學（分段講解與程式碼複製貼上）

> 對象：修習機器學習模型部署與 Web UI 開發的學員
> 對應程式碼：`app.py` 第 192~470 行 (`Gradio UI` 區塊)
>
> 本 Notebook 採用**分段拆解教學法**：每一區塊先講解核心觀念，再提供可以直接複製貼入 `app.py` 的程式碼，並詳細說明「為什麼要這樣寫」。

## 🎯 學習目標

- [ ] 能使用 HTML/CSS 打造現代化的視覺卡片（預測結果卡、評估指標卡、迴歸方程式、特徵影響力圖）
- [ ] 理解 Gradio 事件處理器（Handler）如何接收前端組件輸入並回傳動態 HTML
- [ ] 掌握 Gradio `Blocks` 網頁排版架構（`Tabs`, `Tab`, `Row`, `Column` 雙欄響應式設計）
- [ ] 了解滑桿拖曳時 `.release()` 與 `.change()` 事件對伺服器效能的巨大影響
- [ ] 掌握 Gradio 主題設定與在生產環境（FastAPI + Uvicorn）防止 500 錯誤的必要設定

---
## 📌 步驟 1：HTML 視覺卡片生成函數 (UI Component Builders)

### 💡 核心觀念
Gradio 內建的 `gr.HTML` 允許我們寫原生 HTML/CSS。比起標準的文字輸出，將模型結果包裝成 **現代化儀表板卡片**（漸層背景、陰影、動態顏色、直覺圖條）能極大地提升使用者體驗 (UX)。

### 📋 程式碼貼上區（複製下方程式碼貼入 `app.py` 第 196 行起）：

In [ ]:
# --- 1. 輔助 HTML 生成函數 ---

def make_prediction_card(salary: float) -> str:
    annual = salary * 14
    return f"""
    <div style="background: linear-gradient(135deg, #0f9b0f, #38ef7d); color: white; padding: 25px; border-radius: 15px; text-align: center; box-shadow: 0 8px 20px rgba(0,0,0,0.08); margin-bottom: 20px; transition: all 0.3s ease;">
        <span style="font-size: 0.95rem; font-weight: bold; text-transform: uppercase; letter-spacing: 1.5px; opacity: 0.95;">預測月薪薪資</span>
        <h2 style="font-size: 2.8rem; margin: 10px 0; font-weight: 800; text-shadow: 1px 1px 3px rgba(0,0,0,0.15);">{salary:.2f} <span style="font-size: 1.5rem; font-weight: 400;">k</span></h2>
        <span style="font-size: 1.05rem; font-weight: 500; opacity: 0.9;">估計年薪 (14個月): <strong style="font-size: 1.3rem;">{annual:.1f}</strong> k</span>
    </div>
    """

def make_metrics_card(r2: float, train_time: float, intercept: float, test_size: float, random_state: int, model_type: str = "LinearRegression", alpha: float = 1.0) -> str:
    alpha_info = f" (α={alpha})" if model_type.lower() in ["lasso", "ridge"] else ""
    return f"""
    <div style="display: grid; grid-template-columns: repeat(2, 1fr); gap: 16px; margin-bottom: 20px;">
        <div style="background-color: #f8f9fa; padding: 18px 10px; border-radius: 10px; text-align: center; border: 1px solid #e0e0e0; box-shadow: 0 2px 6px rgba(0,0,0,0.02);">
            <div style="font-size: 0.8rem; color: #5f6368; font-weight: bold; text-transform: uppercase; letter-spacing: 0.5px;">決定係數 R² Score</div>
            <div style="font-size: 2rem; font-weight: 800; color: #1a73e8; margin-top: 5px;">{r2:.4f}</div>
        </div>
        <div style="background-color: #f8f9fa; padding: 18px 10px; border-radius: 10px; text-align: center; border: 1px solid #e0e0e0; box-shadow: 0 2px 6px rgba(0,0,0,0.02);">
            <div style="font-size: 0.8rem; color: #5f6368; font-weight: bold; text-transform: uppercase; letter-spacing: 0.5px;">模型訓練耗時</div>
            <div style="font-size: 2rem; font-weight: 800; color: #137333; margin-top: 5px;">{train_time:.4f}s</div>
        </div>
    </div>
    <div style="font-size: 0.92rem; color: #3c4043; background: #e8f0fe; padding: 12px 18px; border-radius: 8px; border: 1px solid #d2e3fc; font-weight: 600; margin-bottom: 10px;">
        🤖 <strong>模型演算法:</strong> <span style="color: #1a73e8;">{model_type}{alpha_info}</span>
    </div>
    <div style="font-size: 0.92rem; color: #3c4043; background: #f1f3f4; padding: 12px 18px; border-radius: 8px; border: 1px solid #e0e0e0; font-weight: 500; margin-bottom: 10px;">
        🏠 <strong>模型截距 (Intercept / 偏置值 b):</strong> {intercept:.4f}
    </div>
    <div style="font-size: 0.85rem; color: #5f6368; display: flex; justify-content: space-between; font-weight: 500; padding: 0 5px;">
        <span>📊 <strong>測試集比例:</strong> {test_size * 100:.0f}%</span>
        <span>🌱 <strong>隨機種子:</strong> {random_state}</span>
    </div>
    """

def make_equation_html(feature_coefs: dict[str, float], intercept: float) -> str:
    html_parts = []
    for name, coef in feature_coefs.items():
        color = "#137333" if coef >= 0 else "#c5221f"
        sign = "+" if coef >= 0 else "-"
        html_parts.append(f"""
        <span style="white-space: nowrap; margin: 0 4px; display: inline-block;">
            {sign} <strong style="color: {color};">{abs(coef):.3f}</strong> × <span style="color: #202124; font-weight: 600;">({name})</span>
        </span>
        """)
    
    html_eq = f"""
    <div style="background-color: #f8f9fa; border-left: 5px solid #1a73e8; padding: 15px; border-radius: 6px; margin-top: 15px; box-shadow: 0 1px 3px rgba(0,0,0,0.05); border: 1px solid #e0e0e0; border-left: 5px solid #1a73e8;">
        <h4 style="margin: 0 0 8px 0; color: #202124; font-size: 0.95rem; font-weight: 700; text-transform: uppercase; letter-spacing: 0.5px;">🧮 擬合迴歸方程式 (Fitted Equation)</h4>
        <div style="font-family: 'Consolas', 'Courier New', Courier, monospace; font-size: 1.05rem; color: #3c4043; line-height: 1.6; word-wrap: break-word; padding: 5px 0;">
            <strong style="color: #1a73e8;">Salary (預測月薪)</strong> = 
            <span style="font-weight: bold;">{intercept:.3f}</span>
            {" ".join(html_parts)}
        </div>
        <p style="font-size: 0.78rem; color: #5f6368; margin: 8px 0 0 0; line-height: 1.45; border-top: 1px dashed #e0e0e0; padding-top: 8px;">
            *註：方程式中的變數皆為<strong>標準化 (Standardized)</strong> 後的數值。權重為正（<span style="color: #137333; font-weight: bold;">綠色</span>）代表該特徵增加會提升薪資，權重為負（<span style="color: #c5221f; font-weight: bold;">紅色</span>）代表該特徵增加會降低薪資。
        </p>
    </div>
    """
    return html_eq

def make_importance_chart(feature_coefs: dict[str, float]) -> str:
    if not feature_coefs:
        return "<p style='color: #5f6368; text-align: center; padding: 20px;'>目前無特徵權重資料</p>"
    
    sorted_coefs = sorted(feature_coefs.items(), key=lambda x: abs(x[1]), reverse=True)
    max_abs_val = max(abs(val) for val in feature_coefs.values()) if feature_coefs else 1.0
    if max_abs_val == 0:
        max_abs_val = 1.0
        
    html = '<div style="margin-top: 15px; display: flex; flex-direction: column; gap: 14px;">'
    html += '<h4 style="margin: 0 0 8px 0; font-size: 1.1rem; font-weight: 700; color: #202124; letter-spacing: 0.3px;">💡 特徵影響力分析 (Feature Coefficients)</h4>'
    
    for feature, val in sorted_coefs:
        pct = (abs(val) / max_abs_val) * 100
        color = "#137333" if val >= 0 else "#c5221f"
        direction_text = " (正向加薪 📈)" if val >= 0 else " (負向減薪 📉)"
        html += f"""
        <div>
            <div style="display: flex; justify-content: space-between; margin-bottom: 5px; font-weight: 600; font-size: 0.95rem; color: #3c4043;">
                <span>{feature}{direction_text}</span>
                <span style="color: {color}; font-family: monospace;">{val:+.4f}</span>
            </div>
            <div style="background-color: #f1f3f4; border-radius: 8px; height: 12px; overflow: hidden; width: 100%;">
                <div style="background-color: {color}; width: {pct}%; height: 100%; border-radius: 8px; transition: width 0.7s cubic-bezier(0.4, 0, 0.2, 1);"></div>
            </div>
        </div>
        """
    html += '</div>'
    return html

### 🔍 為什麼要這樣寫？（原理拆解）

1. **`make_prediction_card`**：傳入模型算出的月薪，用 Python f-string 自動算出 14 個月估算年薪，並生成雙色漸層卡片。
2. **`make_equation_html`**：自動把特徵與權重組合呈現數學公式：$y = b + w_1 x_1 + w_2 x_2 ...$。若權重為正則標示為**加薪綠色** (`#137333`)，若為負則標示為**減薪紅色** (`#c5221f`)。
3. **`make_importance_chart`**：算出所有特徵權重的絕對值最大值 `max_abs_val`，並用 `pct = (abs(val) / max_abs_val) * 100` 將數值動態轉化為 CSS `width: {pct}%` 長條圖進度條。

---
## 📌 步驟 2：Gradio 事件處理函數 (Event Handlers)

### 💡 核心觀念
網頁上的滑桿、下拉選單或按鈕被觸發時，需要後端 Python 函數接收參數、進行機器學習運算，最後回傳前端要呈現的資料或 HTML 卡片。

### 📋 程式碼貼上區（複製下方程式碼貼入 `app.py` 第 291 行起）：

In [ ]:
# --- 2. Gradio 事件處理器 ---

def predict_gradio_handler(years_exp, edu_level, city):
    """
    處理 Gradio UI 的預測請求：接收 UI 輸入 -> 預處理編碼 -> 模型的預測 -> 生成 HTML 卡片
    """
    oe = MODEL_STATE["oe"]
    ohe = MODEL_STATE["ohe"]
    scaler = MODEL_STATE["scaler"]
    model = MODEL_STATE["model"]
    
    edu_encoded = int(oe.transform(pd.DataFrame([[edu_level]], columns=["EducationLevel"])))[0][0])
    city_encoded = ohe.transform(pd.DataFrame([[city]], columns=["City"])))[0]
    
    feature_names = MODEL_STATE["feature_names"]
    features_df = pd.DataFrame([[years_exp, edu_encoded] + list(city_encoded)], columns=feature_names)
    features_scaled = scaler.transform(features_df)
    
    pred_val = float(model.predict(features_scaled)[0])
    
    return make_prediction_card(pred_val)


def train_gradio_handler(test_size, random_state, model_type, alpha):
    """
    處理 Gradio UI 的重新訓練請求：執行線上訓練 -> 重載狀態 -> 一次元傳 4 個 UI 區塊內容
    """
    res = train_and_save_model(
        test_size=float(test_size),
        random_state=int(random_state),
        model_type=str(model_type),
        alpha=float(alpha)
    )
    
    load_model_state()
    
    metrics_html = make_metrics_card(
        r2=MODEL_STATE["r2"],
        train_time=MODEL_STATE["train_time"],
        intercept=MODEL_STATE["intercept"],
        test_size=MODEL_STATE["test_size"],
        random_state=MODEL_STATE["random_state"],
        model_type=MODEL_STATE.get("model_type", "LinearRegression"),
        alpha=MODEL_STATE.get("alpha", 1.0)
    )
    equation_html = make_equation_html(MODEL_STATE["feature_coefs"], MODEL_STATE["intercept"])
    importance_html = make_importance_chart(MODEL_STATE["feature_coefs"])
    status_text = f"### 📢 最新狀態: `✅ {MODEL_STATE.get('model_type', 'LinearRegression')} 模型線上重新訓練並載入成功！`"
    
    return status_text, metrics_html, equation_html, importance_html

### 🔍 為什麼要這樣寫？（原理拆解）

1. **單一輸入 vs 多重輸出**：`train_gradio_handler` 接受 4 個參數（`test_size`, `random_state`, `model_type`, `alpha`），並同時回傳 4 個物件（`status_text`, `metrics_html`, `equation_html`, `importance_html`），在 Gradio 中可以用一個按鈕同時刷新多個 UI 區域！
2. **全域狀態同步**：重訓完畢後呼叫 `load_model_state()`，確保全域 `MODEL_STATE` 更新，接著立刻用最新 `MODEL_STATE` 生成視覺卡片。

---
## 📌 步驟 3：計算初始 UI 頁面內容 (Initial State)

### 💡 核心觀念
當使用者第一次打開網頁時，如果元件沒有初始內容，卡片處會呈現空白。因此我們在網頁渲染前先用預設參數計算出第一筆 HTML。

### 📋 程式碼貼上區（複製下方程式碼貼入 `app.py` 第 346 行起）：

In [ ]:
# --- 3. 初始 UI 內容計算 ---
initial_pred_card = predict_gradio_handler(5.0, "大學", "城市A")
initial_metrics = make_metrics_card(
    r2=MODEL_STATE["r2"],
    train_time=MODEL_STATE["train_time"],
    intercept=MODEL_STATE["intercept"],
    test_size=MODEL_STATE["test_size"],
    random_state=MODEL_STATE["random_state"],
    model_type=MODEL_STATE.get("model_type", "LinearRegression"),
    alpha=MODEL_STATE.get("alpha", 1.0)
)
initial_equation = make_equation_html(MODEL_STATE["feature_coefs"], MODEL_STATE["intercept"])
initial_importance = make_importance_chart(MODEL_STATE["feature_coefs"])

---
## 📌 步驟 4：Gradio Blocks 布局——分頁一：「即時月薪預測」

### 💡 核心觀念
使用 `gr.Blocks()`（Gradio 的高級自訂版面語法）搭配 `gr.Tabs()`, `gr.Tab()`, `gr.Row()`, `gr.Column()` 進行**雙欄響應式網頁排版**（左欄輸入特徵，右欄輸出卡片）。

### 📋 程式碼貼上區（複製下方程式碼貼入 `app.py` 第 361 行起）：

In [ ]:
# --- 4. 建立 Gradio UI Blocks 布局 ---
import gradio as gr

with gr.Blocks(title="💼 薪資預測多元線性迴歸平台") as demo:
    gr.Markdown(
        """
        # 💼 薪資預測多元線性迴歸教學與部署平台
        本系統展示了機器學習模型部署的**完整生命週期**。此服務底層使用 **FastAPI** 驅動，提供標準化 RESTful API，並結合 **Gradio** 開發了互動式 Web 介面。
        * 🔮 **即時預測分頁**：輸入您的工作年資、學歷與工作城市，即時透過多元線性迴歸模型取得月薪與年薪估計。
        * ⚙️ **線上訓練與公式分頁**：可線上調整測試集切分比例與隨機種子，即時訓練模型，並動態展示擬合後的**數學迴歸方程式**與特徵權重係數。
        """
    )
    
    with gr.Tabs():
        # --- 分頁一：即時預測 ---
        with gr.Tab("🔮 即時月薪預測"):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("### 1. 輸入特徵 (Features)")
                    years_exp = gr.Slider(minimum=1.0, maximum=10.0, value=5.0, step=0.1, label="工作年資 (Years Experience)")
                    edu_level = gr.Dropdown(choices=["大學", "碩士以上", "高中以下"], value="大學", label="教育學歷 (Education Level)")
                    city = gr.Dropdown(choices=["城市A", "城市B", "城市C"], value="城市A", label="工作城市 (City)")
                    predict_btn = gr.Button("🔮 開始預測", variant="primary")
                    
                with gr.Column(scale=1):
                    gr.Markdown("### 2. 預測結果")
                    output_card = gr.HTML(value=initial_pred_card, label="薪資預測卡片")
            
            inputs = [years_exp, edu_level, city]
            
            # 事件綁定：對於 Slider 使用 .release() 避免拖曳時頻繁發送 API 請求
            years_exp.release(fn=predict_gradio_handler, inputs=inputs, outputs=[output_card], queue=False, show_progress="hidden")
            edu_level.change(fn=predict_gradio_handler, inputs=inputs, outputs=[output_card], queue=False, show_progress="hidden")
            city.change(fn=predict_gradio_handler, inputs=inputs, outputs=[output_card], queue=False, show_progress="hidden")
            predict_btn.click(fn=predict_gradio_handler, inputs=inputs, outputs=[output_card], queue=False, show_progress="hidden")

### 🔍 為什麼要這樣寫？（關鍵效能與 UX 優化）

1. **`scale=1` 雙欄對齊**：`gr.Column(scale=1)` 讓左右兩欄在寬螢幕下以 50% : 50% 比例權重對齊，手機等窄螢幕時會自動轉為單欄垂直排列。
2. **⚠️ 關鍵差異：`years_exp.release()` vs `.change()`**：
   - 如果對 `Slider` 使用 `.change()`，使用者在拖動滑桿的 1 秒鐘內，會發送數十次請求給後端，造成伺服器極度卡頓。
   - 使用 `.release()` 只會在使用者**放開滑桿**時觸發一次！而 `Dropdown` 沒有拖動過程，使用 `.change()` 則是即時且安全的。

---
## 📌 步驟 5：Gradio Blocks 布局——分頁二：「線上模型訓練與公式解析」

### 💡 核心觀念
建立第二個分頁 `Tab`（線上模型訓練與公式解析），提供模型選擇與超參數輸入，並將重訓結果一次連動更新到多個輸出卡片。

### 📋 程式碼貼上區（複製下方程式碼貼入 `app.py` 的 `with gr.Tabs():` 區塊內）：

In [ ]:
        # --- 分頁二：線上訓練 ---
        with gr.Tab("⚙️ 線上模型訓練與公式解析"):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("### 1. 調整訓練參數")
                    m_type = gr.Dropdown(
                        choices=["LinearRegression", "Lasso", "Ridge"],
                        value=MODEL_STATE.get("model_type", "LinearRegression"),
                        label="模型演算法 (Model Type)"
                    )
                    alpha_val = gr.Slider(
                        minimum=0.01, maximum=10.0, value=MODEL_STATE.get("alpha", 1.0), step=0.05,
                        label="正則化強度 (alpha / 懲罰項，僅適用 Lasso & Ridge)"
                    )
                    t_size = gr.Slider(minimum=0.1, maximum=0.5, value=MODEL_STATE["test_size"], step=0.05, label="測試集比例 (test_size)")
                    seed = gr.Number(value=MODEL_STATE["random_state"], label="隨機種子 (random_state)", precision=0)
                    
                    train_btn = gr.Button("🚀 開始訓練模型", variant="primary")
                    
                with gr.Column(scale=1):
                    gr.Markdown("### 2. 訓練結果與特徵分析")
                    train_status = gr.Markdown("### 📢 最新狀態: `已載入預訓練模型 (就緒)`")
                    metrics_card = gr.HTML(value=initial_metrics, label="評估指標卡片")
                    equation_box = gr.HTML(value=initial_equation, label="迴歸方程式")
                    importance_chart = gr.HTML(value=initial_importance, label="特徵重要性圖表")
            
            # 綁定訓練按鈕點擊事件：一對多更新 (4 inputs -> 4 outputs)
            train_btn.click(
                fn=train_gradio_handler,
                inputs=[t_size, seed, m_type, alpha_val],
                outputs=[train_status, metrics_card, equation_box, importance_chart],
                queue=False,
                show_progress="minimal",
            )

### 🔍 為什麼要這樣寫？（原理拆解）

1. **按鈕觸發事件**：`train_btn.click` 的 `inputs` 傳入 4 個元件，`outputs` 設定 4 個接收端。後端 `train_gradio_handler` 回傳的 4 個數值會精準填入對應的 UI 位置。
2. **微小加載動畫**：設定 `show_progress='minimal'` 在點擊重訓時只在按鈕周圍顯示動畫，不會遮擋整個畫面。

---
## 📌 步驟 6：主題風格與生產環境 (FastAPI / Uvicorn) 防錯設定

### 💡 核心觀念
在將 Gradio 掛載到 FastAPI 應用中（或部署在雲端如 Render）時，預設的動態 CSS 生成可能會因為主題載入失敗而拋出 500 Internal Server Error。我們需要手動預算主題哈希與設定佇列併發限制。

### 📋 程式碼貼上區（複製下方程式碼貼入 `app.py` 的結尾部分）：

In [ ]:
# --- 6. 主題風格與生產環境防錯設定 ---
import hashlib

# 設定柔和色彩主題
demo.theme = gr.themes.Soft(primary_hue="teal", secondary_hue="indigo")

# 手動計算主題 CSS 與哈希值，防範與 FastAPI / Uvicorn 整合時引發 500 錯誤
demo.theme_css = demo.theme._get_theme_css()
demo.stylesheets = demo.theme._stylesheets
demo.theme_hash = hashlib.sha256(demo.theme_css.encode("utf-8")).hexdigest()

# 設定佇列上限，防止多使用者同時操作時造成阻塞
demo.queue(default_concurrency_limit=10)

---
## 🚀 總結與驗證流程

將步驟 1~6 的程式碼依序複製貼入 `app.py` 後，最後在 FastAPI 中掛載此 Gradio 介面（例如 `app = gr.mount_gradio_app(api_app, demo, path="/")`）。

在終端機中執行：
```bash
uvicorn app:api_app --reload
```

開啟瀏覽器訪問 `http://127.0.0.1:8000`，即可體驗完整的預測與線上模型訓練 Web UI！